# HEMM Enhanced Dataset — Data Cleaning
## NALCO Internship Project
### Dataset: HEMM_Dataset_ENHANCED.csv | Columns: 90 | Rows: 2000
---
**Cleaning Steps:**
1. Load & Inspect
2. Remove Duplicates
3. Fix Date Column
4. Fill Missing Values
5. Fix Outliers using Upper & Lower Limits
6. Encode Text Columns
7. Encode Status & Condition Columns
8. Encode Maintenance Columns
9. Encode Parts/Replacement Columns
10. Save Final Cleaned Dataset

## Step 1 — Load & Inspect the Dataset

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Create output folders
os.makedirs('../data',   exist_ok=True)
os.makedirs('../graphs', exist_ok=True)

# Load dataset
df = pd.read_csv(r'D:\Nalco Intenship\Predictive-analysis-of-IT-OT-equipment-maintainance\HEMM_Dataset_ENHANCED.csv')

print("=" * 55)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 55)
print(f"Total Rows    : {df.shape[0]}")
print(f"Total Columns : {df.shape[1]}")
print(f"Failure Rate  : {df['Failure'].mean()*100:.1f}%")
print(f"Duplicates    : {df.duplicated().sum()}")
print()
print("Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0])


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: malformed \N character escape (742820447.py, line 12)

In [ ]:
# See first 3 rows
df.head(3)


In [ ]:
# Check data types of all columns
print("COLUMN DATA TYPES:")
print("-" * 40)
for col in df.columns:
    print(f"  {col:45s} → {str(df[col].dtype)}")


## Step 2 — Remove Duplicate Rows
**Why:** Duplicate rows confuse the ML model and give wrong accuracy scores.

In [ ]:
before = df.shape[0]
df = df.drop_duplicates()
after  = df.shape[0]

print(f"Rows before : {before}")
print(f"Rows after  : {after}")
print(f"Removed     : {before - after} duplicate rows")


## Step 3 — Fix Date Column
**Why:** Date is stored as plain text. We convert it to a real date and extract Year, Month, Day — so the model can learn time patterns.

In [ ]:
df['Date']        = pd.to_datetime(df['Date'])
df['Year']        = df['Date'].dt.year
df['Month']       = df['Date'].dt.month
df['Day_of_Week'] = df['Date'].dt.dayofweek   # 0=Monday, 6=Sunday

print("Date column fixed!")
print("New columns added: Year, Month, Day_of_Week")
print()
print(df[['Date', 'Year', 'Month', 'Day_of_Week']].head(5))


## Step 4 — Fill Missing Values
**Why:** ML model cannot work with blank/empty cells. We fill them smartly.

- `Failure_Type` and `Failure_Component` are blank when there is NO failure → fill with **'None'**
- Number columns → fill with **median** (safe middle value)

In [ ]:
# --- Text columns with missing values ---
print("Missing BEFORE filling:")
print(df[['Failure_Type','Failure_Component']].isnull().sum())

df['Failure_Type']      = df['Failure_Type'].fillna('None')
df['Failure_Component'] = df['Failure_Component'].fillna('None')

print()
print("Missing AFTER filling:")
print(df[['Failure_Type','Failure_Component']].isnull().sum())
print()
print("Failure_Type values   :", df['Failure_Type'].unique().tolist())
print("Failure_Component values:", df['Failure_Component'].unique().tolist())


In [ ]:
# --- Numeric columns (fill with median if any missing) ---
num_cols = [
    'Engine_Temp_C', 'Oil_Pressure_bar', 'Vibration_mms',
    'Fuel_Consumption_Lhr', 'Tyre_Pressure_PSI', 'Coolant_Level',
    'Battery_Voltage_V', 'Hydraulic_Pressure_bar', 'Exhaust_Temp_C',
    'RPM', 'Operating_Hours', 'Load_Cycles_per_Day',
    'Maintenance_Cost_INR', 'Estimated_Downtime_Hrs',
    'Days_Since_Last_Maintenance', 'Health_Score',
    'Engine_Life_Remaining_pct', 'Tyre_Life_Remaining_pct',
    'Hydraulic_Life_Remaining_pct', 'Battery_Life_Remaining_pct',
    'Brake_Life_Remaining_pct'
]

filled = 0
for col in num_cols:
    missing = df[col].isnull().sum()
    if missing > 0:
        df[col] = df[col].fillna(df[col].median())
        print(f"  Filled {missing} missing in '{col}' with median = {df[col].median():.2f}")
        filled += missing

if filled == 0:
    print("No missing values in numeric columns — all good!")
print(f"Total missing values remaining: {df.isnull().sum().sum()}")


## Step 5 — Fix Outliers Using Upper & Lower Limits
**Why:** We already have Upper & Lower limit columns in the dataset. We use them directly to clip sensor readings to valid ranges. This is better than guessing!

**clip()** means: if value is less than lower limit → set it to lower limit. If more than upper limit → set it to upper limit.

In [ ]:
# Sensor columns and their corresponding limit columns
sensor_limits = {
    'Engine_Temp_C'         : ('Engine_Temp_C_Lower_Limit',          'Engine_Temp_C_Upper_Limit'),
    'Oil_Pressure_bar'      : ('Oil_Pressure_bar_Lower_Limit',        'Oil_Pressure_bar_Upper_Limit'),
    'Vibration_mms'         : ('Vibration_mms_Lower_Limit',           'Vibration_mms_Upper_Limit'),
    'Fuel_Consumption_Lhr'  : ('Fuel_Consumption_Lhr_Lower_Limit',    'Fuel_Consumption_Lhr_Upper_Limit'),
    'Tyre_Pressure_PSI'     : ('Tyre_Pressure_PSI_Lower_Limit',       'Tyre_Pressure_PSI_Upper_Limit'),
    'Coolant_Level'         : ('Coolant_Level_Lower_Limit',           'Coolant_Level_Upper_Limit'),
    'Battery_Voltage_V'     : ('Battery_Voltage_V_Lower_Limit',       'Battery_Voltage_V_Upper_Limit'),
    'Hydraulic_Pressure_bar': ('Hydraulic_Pressure_bar_Lower_Limit',  'Hydraulic_Pressure_bar_Upper_Limit'),
    'Exhaust_Temp_C'        : ('Exhaust_Temp_C_Lower_Limit',          'Exhaust_Temp_C_Upper_Limit'),
    'RPM'                   : ('RPM_Lower_Limit',                     'RPM_Upper_Limit'),
}

print("Checking and fixing outliers using actual upper/lower limits:")
print("-" * 60)

for sensor, (lo_col, hi_col) in sensor_limits.items():
    lo = df[lo_col].iloc[0]   # lower limit value
    hi = df[hi_col].iloc[0]   # upper limit value

    below = (df[sensor] < lo).sum()
    above = (df[sensor] > hi).sum()

    if below > 0 or above > 0:
        df[sensor] = df[sensor].clip(lo, hi)
        print(f"  {sensor:30s} | Below limit: {below} | Above limit: {above} | Fixed!")
    else:
        print(f"  {sensor:30s} | No outliers found ✓")

print()
print("Outlier fixing complete!")


## Step 6 — Encode Main Text Columns
**Why:** ML model only understands numbers, not words. We convert every text column to numbers using LabelEncoder.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# --- Equipment & operational columns ---
main_text_cols = [
    'Equipment_Type',
    'Failure_Type',
    'Failure_Component',
    'Maintenance_Priority',
    'Maintenance_Type',
    'PM_Status',
    'Last_Maintenance_Type',
    'Maintenance_Team',
]

print("Encoding main text columns:")
print("-" * 55)
for col in main_text_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col])
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"  {col}")
    print(f"    → {mapping}")
    print()


## Step 7 — Encode Status Columns (Sensor Status)
**Why:** Every sensor has a Status column like 'Normal', 'Above Normal', 'Below Normal'. We convert these to numbers.

- Normal       → 0
- Above Normal → 1  (too high — warning)
- Below Normal → -1 (too low — warning)

In [ ]:
# Status columns for all 10 sensors
status_cols = [
    'Engine_Temp_C_Status',
    'Oil_Pressure_bar_Status',
    'Vibration_mms_Status',
    'Fuel_Consumption_Lhr_Status',
    'Tyre_Pressure_PSI_Status',
    'Coolant_Level_Status',
    'Battery_Voltage_V_Status',
    'Hydraulic_Pressure_bar_Status',
    'Exhaust_Temp_C_Status',
    'RPM_Status',
]

status_map = {
    'Normal'       :  0,
    'Above Normal' :  1,
    'Below Normal' : -1
}

print("Encoding sensor status columns:")
print("-" * 45)
for col in status_cols:
    new_col = col.replace('_Status', '_Status_enc')
    df[new_col] = df[col].map(status_map)
    print(f"  {col:40s} → {new_col}")

print()
print("Status encoding mapping used:")
print(f"  Normal = 0 | Above Normal = 1 | Below Normal = -1")


## Step 8 — Encode Parts Condition Columns
**Why:** Part condition columns have Good / Warning / Critical — we convert to numbers.

- Good     → 0 (healthy)
- Warning  → 1 (needs attention)
- Critical → 2 (needs immediate action)

In [ ]:
# Part condition columns
condition_cols = [
    'Engine_Condition',
    'Tyre_Condition',
    'Hydraulic_Condition',
    'Brake_Condition',
    'Electrical_Condition',
    'Fuel_System_Condition',
    'Transmission_Condition',
    'Cooling_System_Condition',
]

condition_map = {
    'Good'     : 0,
    'Warning'  : 1,
    'Critical' : 2
}

print("Encoding part condition columns:")
print("-" * 50)
for col in condition_cols:
    new_col = col + '_enc'
    df[new_col] = df[col].map(condition_map)
    counts = df[col].value_counts().to_dict()
    print(f"  {col:30s} → {counts}")

print()
print("Condition encoding: Good=0 | Warning=1 | Critical=2")


## Step 9 — Encode Replacement Needed Columns
**Why:** Replacement columns have Yes / Monitor / No — convert to numbers.

- No      → 0
- Monitor → 1
- Yes     → 2

In [ ]:
replacement_cols = [
    'Engine_Replacement_Needed',
    'Tyre_Replacement_Needed',
    'Hydraulic_Replacement_Needed',
    'Brake_Replacement_Needed',
    'Electrical_Replacement_Needed',
]

replacement_map = {
    'No'      : 0,
    'Monitor' : 1,
    'Yes'     : 2
}

print("Encoding replacement needed columns:")
print("-" * 50)
for col in replacement_cols:
    new_col = col + '_enc'
    df[new_col] = df[col].map(replacement_map)
    print(f"  {col:40s} → done")

print()
print("Replacement encoding: No=0 | Monitor=1 | Yes=2")


## Step 10 — Select Final Columns & Save
**Why:** We now select only the columns needed for the ML model. We keep:
- Original sensor readings
- Encoded text columns (ending in _enc)
- Limit columns (Upper/Lower)
- Life remaining columns
- Target column: Failure

In [ ]:
# Final columns for ML model
final_cols = [
    # Identity (keep for reference — not used in model)
    'Equipment_ID', 'Equipment_Type', 'Date',

    # Encoded equipment info
    'Equipment_Type_enc', 'Shift_enc', 'Road_Condition_enc',
    'Operator_Experience_Yr',

    # Raw sensor readings
    'Engine_Temp_C', 'Oil_Pressure_bar', 'Vibration_mms',
    'Fuel_Consumption_Lhr', 'Tyre_Pressure_PSI', 'Coolant_Level',
    'Battery_Voltage_V', 'Hydraulic_Pressure_bar',
    'Exhaust_Temp_C', 'RPM',

    # Upper & Lower limits for each sensor
    'Engine_Temp_C_Lower_Limit',         'Engine_Temp_C_Upper_Limit',
    'Oil_Pressure_bar_Lower_Limit',      'Oil_Pressure_bar_Upper_Limit',
    'Vibration_mms_Lower_Limit',         'Vibration_mms_Upper_Limit',
    'Fuel_Consumption_Lhr_Lower_Limit',  'Fuel_Consumption_Lhr_Upper_Limit',
    'Tyre_Pressure_PSI_Lower_Limit',     'Tyre_Pressure_PSI_Upper_Limit',
    'Coolant_Level_Lower_Limit',         'Coolant_Level_Upper_Limit',
    'Battery_Voltage_V_Lower_Limit',     'Battery_Voltage_V_Upper_Limit',
    'Hydraulic_Pressure_bar_Lower_Limit','Hydraulic_Pressure_bar_Upper_Limit',
    'Exhaust_Temp_C_Lower_Limit',        'Exhaust_Temp_C_Upper_Limit',
    'RPM_Lower_Limit',                   'RPM_Upper_Limit',

    # Sensor status encoded
    'Engine_Temp_C_Status_enc',          'Oil_Pressure_bar_Status_enc',
    'Vibration_mms_Status_enc',          'Fuel_Consumption_Lhr_Status_enc',
    'Tyre_Pressure_PSI_Status_enc',      'Coolant_Level_Status_enc',
    'Battery_Voltage_V_Status_enc',      'Hydraulic_Pressure_bar_Status_enc',
    'Exhaust_Temp_C_Status_enc',         'RPM_Status_enc',

    # Operational info
    'Operating_Hours', 'Load_Cycles_per_Day',
    'Temp_Oil_Ratio', 'Vib_RPM_Ratio',
    'Health_Score', 'High_Risk',
    'Year', 'Month', 'Day_of_Week',

    # Maintenance info encoded
    'Maintenance_Type_enc', 'PM_Interval_Hours',
    'Next_PM_Due_Hours', 'PM_Status_enc',
    'Maintenance_Cost_INR', 'Estimated_Downtime_Hrs',
    'Last_Maintenance_Type_enc', 'Days_Since_Last_Maintenance',
    'Maintenance_Team_enc',

    # Part conditions encoded
    'Engine_Condition_enc',       'Tyre_Condition_enc',
    'Hydraulic_Condition_enc',    'Brake_Condition_enc',
    'Electrical_Condition_enc',   'Fuel_System_Condition_enc',
    'Transmission_Condition_enc', 'Cooling_System_Condition_enc',

    # Replacement needed encoded
    'Engine_Replacement_Needed_enc',    'Tyre_Replacement_Needed_enc',
    'Hydraulic_Replacement_Needed_enc', 'Brake_Replacement_Needed_enc',
    'Electrical_Replacement_Needed_enc',

    # Part life remaining
    'Engine_Life_Remaining_pct',   'Tyre_Life_Remaining_pct',
    'Hydraulic_Life_Remaining_pct','Battery_Life_Remaining_pct',
    'Brake_Life_Remaining_pct',

    # Label columns (for display, not model input)
    'Failure_Type', 'Failure_Type_enc',
    'Failure_Component', 'Failure_Component_enc',
    'Maintenance_Priority', 'Maintenance_Priority_enc',
    'Spare_Parts_Required', 'Prescriptive_Action',
    'Days_to_Next_Failure',

    # TARGET column
    'Failure'
]

df_final = df[final_cols].copy()

print("=" * 55)
print("CLEANING COMPLETE!")
print("=" * 55)
print(f"Final Rows      : {df_final.shape[0]}")
print(f"Final Columns   : {df_final.shape[1]}")
print(f"Missing Values  : {df_final.isnull().sum().sum()}")
print(f"Failure Rate    : {df_final['Failure'].mean()*100:.1f}%")

# Save
df_final.to_csv('../data/HEMM_Dataset_FINAL.csv', index=False)
print()
print("✅ File saved as: HEMM_Dataset_FINAL.csv")
print("✅ Ready for ML Model building!")


## Cleaning Summary

In [ ]:
print("=" * 60)
print("HEMM ENHANCED DATASET — CLEANING SUMMARY")
print("=" * 60)

summary = {
    "Original columns"             : 90,
    "Final columns"                : df_final.shape[1],
    "Total rows"                   : df_final.shape[0],
    "Duplicates removed"           : 0,
    "Missing values filled"        : 1644,
    "Outliers fixed using limits"  : "10 sensors checked & clipped",
    "Text columns encoded"         : 8,
    "Status columns encoded"       : 10,
    "Condition columns encoded"    : 8,
    "Replacement columns encoded"  : 5,
    "Missing values remaining"     : df_final.isnull().sum().sum(),
    "Ready for ML model"           : "YES ✅"
}

for key, val in summary.items():
    print(f"  {key:40s} : {val}")

print()
print("Encoding Reference:")
print("  Status  → Normal=0 | Above Normal=1 | Below Normal=-1")
print("  Cond.   → Good=0   | Warning=1      | Critical=2")
print("  Replace → No=0     | Monitor=1      | Yes=2")
